In [ ]:
import numpy as np
import pandas as pd 
import torch

from collections import defaultdict
from importlib import import_module
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader
from tqdm import tqdm

from skp.toolbox.functions import load_model_from_config

In [ ]:
cfg_name = "mammo.cfg_ddsm_pretrained_aux_losses"
cfg = import_module(f"skp.configs.{cfg_name}").cfg
weights_path = cfg.save_dir + cfg_name + "/ce2bf46d/fold0/checkpoints/last.ckpt"
model = load_model_from_config(cfg, weights_path, device="cuda:0", eval_mode=True)
dataset = import_module(f"skp.datasets.{cfg.dataset}").Dataset(cfg, mode="val")

In [ ]:
for v in model.named_buffers():
    print(v)

In [ ]:
loader = DataLoader(dataset, batch_size=16, shuffle=False, num_workers=13, drop_last=False)

In [ ]:
pred_dict = defaultdict(list)
for batch in tqdm(loader, total=len(loader)):
    x = batch["x"]
    x = x.to("cuda:0")
    with torch.inference_mode():
        p = model({"x": x})
    pred_dict["y_pred"].append(p["logits"][:, 0].cpu().numpy())
    pred_dict["y_true"].append(batch["y"][:, 0].numpy())
    pred_dict["group_index"].append(batch["group_index"].numpy())

In [ ]:
pred_df_dict = {key: np.concatenate(val) for key, val in pred_dict.items()}
pred_df = pd.DataFrame(pred_df_dict)
pred_df["y_pred_sigmoid"] = pred_df.y_pred.apply(lambda x: 1 / (1 + np.exp(-x)))
pred_df

In [ ]:
pred_df_grouped_mean = pred_df.groupby("group_index").mean()
print(roc_auc_score(pred_df_grouped_mean.y_true, pred_df_grouped_mean.y_pred))
print(roc_auc_score(pred_df_grouped_mean.y_true, pred_df_grouped_mean.y_pred_sigmoid))

In [ ]:
pred_df_grouped_mean = pred_df.groupby("group_index").mean()
print(roc_auc_score(pred_df_grouped_mean.y_true, pred_df_grouped_mean.y_pred))
print(roc_auc_score(pred_df_grouped_mean.y_true, pred_df_grouped_mean.y_pred_sigmoid))